Cargamos los datos en un dataframe de Pandas

In [23]:
import pandas as pd

tracklog = pd.read_parquet("../data/tracklog.parquet")

Eliminamos la columna geometry y los registros con timestamp nulo

In [24]:
tracklog = tracklog.drop(columns=['geometry'])
tracklog['time'] = pd.to_datetime(tracklog['time'], errors='coerce')
# df = df.dropna(subset=['timestamp'])
# df = df[df['timestamp'].notna()]
tracklog.dropna(subset=['time'], inplace=True)

Generamos un track UUID y ordenamos

In [25]:
import uuid

NAMESPACE_BASE = uuid.uuid5(uuid.NAMESPACE_DNS, 'tracklog')
'''
# Esta opción con apply (fila por fila) es más lenta.
df['track_uid'] = df.apply(
    lambda row: str(uuid.uuid5(
        NAMESPACE_BASE, 
        f"{row['source']}::{row['source_file']}::{row['track_fid']}"
    )), 
    axis=1
)    
'''
tracklog['track_uid'] = [str(uuid.uuid5(NAMESPACE_BASE, val)) for val in tracklog['source'].astype(str) + "::" + tracklog['source_file'].astype(str) + "::" + tracklog['track_fid'].astype(str)]
tracklog.sort_values(by=['track_uid', 'time'], inplace=True)

Para calcular la diferencia de tiempo y la distancia geográfica entre puntos consecutivos dentro de cada track_uid, primero debemos asegurarnos de que el DataFrame esté ordenado por track_uid y time. Luego usaremos shift() agrupado o la fórmula Haversine (o GeoPy/PyProj) para la distancia.

1. Diferencia de tiempo (time_delta)
Para restar la hora del punto anterior dentro del mismo track_uid:

In [26]:
tracklog['time_delta'] = tracklog.groupby('track_uid')['time'].diff()

2. Distancia geográfica (distance_meters)
Dado que las coordenadas de latitud y longitud están en una superficie esférica (Tierra), la forma recomendada para calcular la distancia en metros entre cada punto y el anterior es la Fórmula de Haversine:

In [27]:
import numpy as np

def haversine_np(lat1, lon1, lat2, lon2):
    """
    Calcula la distancia Haversine en metros entre dos conjuntos de coordenadas.
    """
    R = 6371000.0  # Radio medio de la Tierra en metros

    lat1_rad, lon1_rad = np.radians(lat1), np.radians(lon1)
    lat2_rad, lon2_rad = np.radians(lat2), np.radians(lon2)

    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    a = np.sin(dlat / 2.0)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2.0)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return R * c

# 1. Obtener la latitud y longitud del punto anterior por cada track_uid
tracklog['prev_lat'] = tracklog.groupby('track_uid')['lat'].shift(1)
tracklog['prev_lon'] = tracklog.groupby('track_uid')['lon'].shift(1)

# 2. Calcular la distancia respecto al punto anterior
tracklog['distance_meters'] = haversine_np(
    tracklog['prev_lat'], tracklog['prev_lon'], 
    tracklog['lat'], tracklog['lon']
)

# 3. Limpiar las columnas auxiliares
tracklog = tracklog.drop(columns=['prev_lat', 'prev_lon'])

Creamos un dataframe resumen de datos por cada track

In [28]:
import hashlib
import pandas as pd

def calc_track_points_md5(timestamps):
    # Unir todos los timestamps ordenados/secuenciales en una sola cadena
    joined = ','.join(timestamps.astype(str))
    return hashlib.md5(joined.encode('utf-8')).hexdigest()

def track_summary(df:pd.DataFrame) -> pd.DataFrame:
    summary = (
        df.groupby('track_uid', as_index=False)
        .agg(
            start_time=('time', 'min'),
            end_time=('time', 'max'),
            distance_m=('distance_meters', 'sum'),
            points=('time', 'count')  # o cualquier columna no nula
        )
    )

    # Calcular la duración total en minutos a partir de la diferencia entre start_time y end_time
    summary['duration_m'] = (summary['end_time'] - summary['start_time']).dt.total_seconds() / 60.0

    # Calculamos velocidad media del track
    summary['speed_kmh'] = summary['distance_m']/(summary['duration_m']*60) * 3.6

    # Reordenar las columnas en el orden exacto deseado
    summary = summary[['track_uid', 'points', 'start_time', 'end_time', 'duration_m', 'distance_m', 'speed_kmh']]

    #1. Asegurar la representación exacta en formato string de las columnas
    summary_strings = (
        summary['start_time'].astype(str) + "::" +
        summary['end_time'].astype(str) + "::" +
        summary['points'].astype(str)
    )

    # 2. Generar el hash MD5 usando list comprehension (muy rápido)
    summary['summary_digest'] = [
        hashlib.md5(val.encode('utf-8')).hexdigest() 
        for val in summary_strings
    ]

    # Calcular la Serie con los digests por track_uid
    points_md5_series = df.groupby('track_uid')['time'].agg(calc_track_points_md5)

    # Asignar a summary
    summary['content_digest'] = summary['track_uid'].map(points_md5_series)

    return summary

summary = track_summary(tracklog)

Eliminamos del tracklog todos aquellos tracks duplicados en base al "content_digest"

In [29]:
# 1. Obtener la lista/máscara de 'track_uid' que queremos CONSERVAR
# Identificamos las filas no duplicadas en 'summary' según 'content_digest'
# (keep='first' conserva el primer track_uid encontrado para cada digest)
summary_sin_duplicados = summary.drop_duplicates(subset=['content_digest'], keep='first')

# 2. Filtrar 'tracklog' manteniendo solo los track_uid válidos
tracklog = tracklog[tracklog['track_uid'].isin(summary_sin_duplicados['track_uid'])]

# 3. Limpiar también 'summary' para que quede alineado con 'tracklog'
summary = summary_sin_duplicados.reset_index(drop=True)

Para detectar si un track está contenido dentro de otro (subtrack temporal/secuencial dentro del mismo contexto), la lógica se basa en comparar las ventanas de tiempo: start_time y end_time.

Un track A es considerado un subtrack de un track B si:

start_time_A >= start_time_B

end_time_A <= end_time_B

Ambas condiciones no son identidades simultáneas (es decir, no es el mismo track comparándose consigo mismo).

Solución mediante Cross-Join / Cross-Merge en summary
La forma más rápida y expresiva en Pandas es realizar un cruce sobre el DataFrame summary:

In [30]:
import pandas as pd

# 1. Realizar un merge cruzado (o cartesiano) de summary consigo mismo
# En Pandas 2.x se utiliza how='cross'
cross_summary = summary.merge(
    summary, 
    how='cross', 
    suffixes=('_child', '_parent')
)

# 2. Excluir la autocomparación (un track no es subtrack de sí mismo)
cross_summary = cross_summary[
    cross_summary['track_uid_child'] != cross_summary['track_uid_parent']
]

# 3. Aplicar las condiciones temporales para identificar subtracks
# (Empieza igual o después Y termina igual o antes que el parent)
is_subtrack_condition = (
    (cross_summary['start_time_child'] >= cross_summary['start_time_parent']) &
    (cross_summary['end_time_child'] <= cross_summary['end_time_parent'])
)

subtracks_detected = cross_summary[is_subtrack_condition]

# 4. Obtener la lista única de 'track_uid' que son subtracks de algún otro track
subtrack_uids = subtracks_detected['track_uid_child'].unique()

# A. Marcar en summary qué tracks son subtracks y de qué track dependen
# summary['is_subtrack'] = summary['track_uid'].isin(subtrack_uids)

# B. Si prefieres eliminar los subtracks de ambos DataFrames:
summary = summary[~summary['track_uid'].isin(subtrack_uids)].reset_index(drop=True)
tracklog = tracklog[tracklog['track_uid'].isin(summary['track_uid'])].reset_index(drop=True)

print(f"Subtracks eliminados: {len(subtrack_uids)}")


Subtracks eliminados: 74


## Deteccion de splits

Dividir tracks en basa a determinados criterios de tiempo o espacio

In [31]:
import pandas as pd

# 1. Asegurar que las métricas por punto están calculadas
# (Ignoramos los valores NaN/0 de los primeros puntos de cada track para no sesgar el cálculo)
points_clean = tracklog[tracklog['time_delta'].dt.total_seconds() > 0].copy()

# 2. Construir la tabla de estadísticas avanzadas por track
track_stats = (
    points_clean.groupby('track_uid', as_index=False)
    .agg(
        # Métricas de Puntos y Tiempo
        total_points=('time', 'count'),
        time_mean_sec=('time_delta', 'mean'),
        time_median_sec=('time_delta', 'median'),
        time_max_sec=('time_delta', 'max'),
        time_std_sec=('time_delta', 'std'),
        time_p95_sec=('time_delta', lambda x: x.quantile(0.95)),
        
        # Métricas de Distancia
        dist_mean_m=('distance_meters', 'mean'),
        dist_median_m=('distance_meters', 'median'),
        dist_max_m=('distance_meters', 'max'),
        dist_p95_m=('distance_meters', lambda x: x.quantile(0.95)),
        
        # Métricas de Velocidad (para detectar saltos anómalos/imposibles)
        # speed_max_kmh=('speed_kmh', 'max'),
        # speed_p95_kmh=('speed_kmh', lambda x: x.quantile(0.95))
    )
)

# 3. Formatear y rellenar nulos en tracks con un solo punto
track_stats = track_stats.fillna(0.0)

# Unir summary con track_stats a través de track_uid
summary = summary.merge(
    track_stats, 
    on='track_uid', 
    how='left'
)


Marcamos candidato a split en aquellos tracks cuyo delta time máximo sea mayor que la mediana multiplicada * 1000

In [32]:
#summary[summary['time_max_sec'].dt.total_seconds() > 3600]
split_candidates = summary[summary['time_max_sec'].dt.total_seconds() > (summary['time_median_sec'].dt.total_seconds() * 1000)]

# 1. Filtrar tracklog dejando solo los candidatos
tracklog = tracklog[tracklog['track_uid'].isin(split_candidates['track_uid'])]

# 2. Obtener los índices exactos del punto con time_delta máximo en cada track
idx_breakpoints = tracklog.groupby('track_uid')['time_delta'].idxmax()

# 3. Inicializar la columna is_break a False y marcar True en esos índices
tracklog['is_break'] = False
tracklog.loc[idx_breakpoints, 'is_break'] = True


In [33]:

# 1. Definir los umbrales de ruptura
UMBRAL_TIEMPO_SEG = 3600  # 1 hora en segundos
UMBRAL_DISTANCIA_M = 5000 # Opcional: p.ej. 5 km de salto sin registros

# 2. Crear una máscara booleana que identifique cuándo OCURRE una ruptura
# (Se evalúa sobre la columna time_delta_sec o el cálculo de tiempo entre puntos)
#is_time_break = tracklog['time_delta'].dt.total_seconds() > UMBRAL_TIEMPO_SEG
#is_dist_break = tracklog['distance_meters'] > UMBRAL_DISTANCIA_M

# Condición combinada de ruptura
# is_break = is_time_break | is_dist_break
# is_break = is_time_break

tracklog['is_break'] = (tracklog['time_delta'].dt.total_seconds() > 3600) | tracklog['is_break']


# 2. Agrupar la serie booleana por track_uid y hacer cumsum()
tracklog['segment_id'] = is_break.groupby(tracklog['track_uid']).cumsum()

# 4. Reconstruir un nuevo 'track_uid' combinando el original y el segmento
tracklog['new_track_uid'] = (
    tracklog['track_uid'].astype(str) + "_seg" + tracklog['segment_id'].astype(str)
)